In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'


In [6]:
# Load the dataset
df = pd.read_csv("train.csv")

# Basic dataset info
print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


Dataset shape: (1804874, 45)
Memory usage: 1486.33 MB


In [ ]:
# Dataset overview - missing values and basic stats
print("Missing values per column:")
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({'Count': missing_counts, 'Percentage': missing_pct})
print(missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False))

print(f"\nBasic statistics for target variable:")
print(df['target'].describe())


In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of target values
axes[0].hist(df['target'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[0].set_title('Distribution of Toxicity Scores')
axes[0].set_xlabel('Toxicity Score')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

# Binary classification threshold analysis
binary_target = (df['target'] >= 0.5).astype(int)
axes[1].bar(['Non-toxic (< 0.5)', 'Toxic (>= 0.5)'], 
           [sum(binary_target == 0), sum(binary_target == 1)], 
           color=['lightgreen', 'lightcoral'])
axes[1].set_title('Binary Classification (threshold=0.5)')
axes[1].set_ylabel('Count')

# Add percentage labels
total = len(df)
for i, v in enumerate([sum(binary_target == 0), sum(binary_target == 1)]):
    axes[1].text(i, v + 100, f'{v/total*100:.1f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"Class imbalance ratio (toxic:non-toxic) = 1:{sum(binary_target == 0)/sum(binary_target == 1):.1f}")


In [ ]:
# Text length analysis
df['comment_length'] = df['comment_text'].str.len()
df['word_count'] = df['comment_text'].str.split().str.len()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Comment length distribution
axes[0,0].hist(df['comment_length'], bins=50, alpha=0.7, color='lightblue', edgecolor='black')
axes[0,0].set_title('Distribution of Comment Length (characters)')
axes[0,0].set_xlabel('Character Count')
axes[0,0].set_ylabel('Frequency')
axes[0,0].set_xlim(0, df['comment_length'].quantile(0.95))  # Remove extreme outliers for better viz

# Word count distribution
axes[0,1].hist(df['word_count'], bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
axes[0,1].set_title('Distribution of Word Count')
axes[0,1].set_xlabel('Word Count')
axes[0,1].set_ylabel('Frequency')
axes[0,1].set_xlim(0, df['word_count'].quantile(0.95))

# Length vs toxicity
toxic_mask = df['target'] >= 0.5
axes[1,0].boxplot([df[~toxic_mask]['comment_length'], df[toxic_mask]['comment_length']], 
                  labels=['Non-toxic', 'Toxic'])
axes[1,0].set_title('Comment Length by Toxicity')
axes[1,0].set_ylabel('Character Count')

# Word count vs toxicity
axes[1,1].boxplot([df[~toxic_mask]['word_count'], df[toxic_mask]['word_count']], 
                  labels=['Non-toxic', 'Toxic'])
axes[1,1].set_title('Word Count by Toxicity')
axes[1,1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()

print(f"Average comment length: {df['comment_length'].mean():.1f} characters")
print(f"Average word count: {df['word_count'].mean():.1f} words")


In [ ]:
# Toxicity categories correlation analysis
toxicity_cols = ['target', 'severe_toxicity', 'obscene', 'identity_attack', 'insult', 'threat']
toxicity_data = df[toxicity_cols].fillna(0)  # Fill NaN with 0 for correlation

# Correlation heatmap using matplotlib
plt.figure(figsize=(8, 6))
corr_matrix = toxicity_data.corr()
im = plt.imshow(corr_matrix, cmap='RdYlBu_r', aspect='auto', vmin=-1, vmax=1)
plt.colorbar(im, label='Correlation')

# Add correlation values as text
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        plt.text(j, i, f'{corr_matrix.iloc[i, j]:.3f}', 
                ha='center', va='center', fontsize=10)

plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha='right')
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title('Correlation Matrix: Toxicity Categories')
plt.tight_layout()
plt.show()

# Distribution of toxicity categories
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(toxicity_cols):
    data = df[col].dropna()
    axes[i].hist(data, bins=30, alpha=0.7, edgecolor='black')
    axes[i].set_title(f'{col.replace("_", " ").title()}')
    axes[i].set_xlabel('Score')
    axes[i].set_ylabel('Frequency')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Identity group analysis
identity_cols = ['asian', 'atheist', 'bisexual', 'black', 'buddhist', 'christian', 
                'female', 'heterosexual', 'hindu', 'homosexual_gay_or_lesbian', 
                'jewish', 'latino', 'male', 'muslim', 'transgender', 'white']

# Count non-null values for each identity group
identity_counts = {}
for col in identity_cols:
    if col in df.columns:
        non_null_count = df[col].notna().sum()
        identity_counts[col] = non_null_count

# Plot identity group mentions
plt.figure(figsize=(12, 6))
sorted_identities = sorted(identity_counts.items(), key=lambda x: x[1], reverse=True)
names, counts = zip(*sorted_identities)

plt.bar(range(len(names)), counts, color='lightcoral', alpha=0.7)
plt.title('Identity Group Mentions in Comments')
plt.xlabel('Identity Groups')
plt.ylabel('Number of Comments')
plt.xticks(range(len(names)), names, rotation=45, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Average toxicity by identity group (for groups with sufficient data)
print("\nAverage toxicity score by identity group (>100 mentions):")
for col in identity_cols:
    if col in df.columns and df[col].notna().sum() > 100:
        avg_toxicity = df[df[col].notna()]['target'].mean()
        print(f"{col.replace('_', ' ').title()}: {avg_toxicity:.3f}")


In [ ]:
# Engagement metrics analysis
engagement_cols = ['funny', 'wow', 'sad', 'likes', 'disagree']
engagement_data = df[engagement_cols].fillna(0)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(engagement_cols):
    # Remove extreme outliers for better visualization
    data = engagement_data[col]
    q95 = data.quantile(0.95)
    filtered_data = data[data <= q95]
    
    axes[i].hist(filtered_data, bins=30, alpha=0.7, edgecolor='black')
    axes[i].set_title(f'{col.title()} Distribution')
    axes[i].set_xlabel('Count')
    axes[i].set_ylabel('Frequency')
    axes[i].grid(True, alpha=0.3)

# Engagement vs toxicity scatter plot
axes[5].scatter(df['target'], df['disagree'], alpha=0.5, s=10)
axes[5].set_title('Toxicity vs Disagree Reactions')
axes[5].set_xlabel('Toxicity Score')
axes[5].set_ylabel('Disagree Count')
axes[5].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Correlation between engagement and toxicity
print("Correlation between engagement metrics and toxicity:")
for col in engagement_cols:
    corr = df['target'].corr(df[col])
    print(f"{col.title()}: {corr:.3f}")


In [ ]:
# Publication and temporal analysis
df['created_date'] = pd.to_datetime(df['created_date'])
df['year'] = df['created_date'].dt.year
df['month'] = df['created_date'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Comments over time
monthly_counts = df.groupby(['year', 'month']).size().reset_index(name='count')
monthly_counts['date'] = pd.to_datetime(monthly_counts[['year', 'month']].assign(day=1))

axes[0].plot(monthly_counts['date'], monthly_counts['count'], marker='o', linewidth=2)
axes[0].set_title('Comments Over Time')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Number of Comments')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Toxicity over time
monthly_toxicity = df.groupby(['year', 'month'])['target'].mean().reset_index()
monthly_toxicity['date'] = pd.to_datetime(monthly_toxicity[['year', 'month']].assign(day=1))

axes[1].plot(monthly_toxicity['date'], monthly_toxicity['target'], marker='o', linewidth=2, color='red')
axes[1].set_title('Average Toxicity Over Time')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Average Toxicity Score')
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"Date range: {df['created_date'].min()} to {df['created_date'].max()}")
print(f"Number of unique publications: {df['publication_id'].nunique()}")


In [ ]:
# Summary statistics
print("=== DATASET SUMMARY ===")
print(f"Total comments: {len(df):,}")
print(f"Toxic comments (>=0.5): {sum(df['target'] >= 0.5):,} ({sum(df['target'] >= 0.5)/len(df)*100:.1f}%)")
print(f"Average toxicity score: {df['target'].mean():.3f}")
print(f"Median comment length: {df['comment_length'].median():.0f} characters")
print(f"Most common publication ID: {df['publication_id'].mode().iloc[0]}")
print(f"Comments with identity annotations: {df[identity_cols].notna().any(axis=1).sum():,}")
print(f"Comments with engagement data: {df[engagement_cols].sum(axis=1).gt(0).sum():,}")
